# Decoder Prediction Error Analysis

This notebook analyzes CSV logs produced by `play_exp_decoder_eval.py`.

It computes decoder prediction errors from successful timesteps only, grouped by:

- terrain
- disturbance condition
- terrain and disturbance condition
- all loaded results pooled together

The metrics cover the full privileged observation, domain randomization variables, specific domain randomization categories, standard observation reconstruction, base velocity, base height, terrain shape around the feet, feet contact/height, and ground reaction forces.

Expected filename pattern:

```text
{task}_{terrain}_{disturbance_condition}_decoder_eval.csv
```

Example:

```text
go1_pact_rough_payload_decoder_eval.csv
go1_abl3_plane_none_decoder_eval.csv
```


In [ ]:
import ast
import math
import os
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
except ImportError:
    sns = None


## Configuration

In [ ]:
# --- User settings ---
EXP_FOLDER = "exp_data_corl_10/recon_01"
TASK_PREFIX = "go1_pact"

# Set to None to load every disturbance condition for this task.
# Otherwise use strings like "none", "payload", "push", "wrench", etc.
DISTURBANCE_CONDITION = None

# Optional filename suffix used by play_exp_decoder_eval.py.
FILENAME_SUFFIX = "_decoder_eval"

RESULTS_DIR = Path(EXP_FOLDER) / "decoder_prediction_analysis_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Chunked CSV reading keeps worker memory bounded. Increase if your files are tiny.
CHUNKSIZE = 512
NUM_WORKERS = min(20, os.cpu_count() or 1)

# Only the three logger columns below are loaded. This avoids materializing full eval logs.
USECOLS = ["failure", "decoder_pred_priv_obs", "gt_priv_obs_latest"]

EPS = 1e-12
EXCLUDE_DIRS = ["analysis_results", "decoder_prediction_analysis_results"]


## Privileged Observation Slices

These slices match the current `go1_pact.py` privileged observation concatenation. The final terrain-height slice is adaptive because some configs/comments report different total decoder output sizes; any dimensions after index 145 are treated as measured terrain heights when present.


In [ ]:
BASE_SLICES = {
    "entire_privileged_observation": (0, None),
    "standard_observation_vector": (0, 57),
    "base_velocity": (57, 60),
    "domain_randomization_all": (95, 145),
    "domain_friction": (95, 96),
    "domain_added_mass": (96, 97),
    "domain_com_shift": (97, 100),
    "domain_pushes": (100, 103),
    "domain_wrenches": (103, 106),
    "base_height": (60, 61),
    "terrain_shape_around_feet": (73, 85),
    "feet_contact_state_and_height": (85, 93),
    "feet_contact_state": (85, 89),
    "feet_height": (89, 93),
    "ground_reaction_forces": (61, 73),
}


def effective_slices(num_dims):
    slices = {}
    for name, (start, stop) in BASE_SLICES.items():
        stop_eff = num_dims if stop is None else min(stop, num_dims)
        if start < num_dims and stop_eff > start:
            slices[name] = slice(start, stop_eff)

    if num_dims > 145:
        slices["terrain_height_measurements"] = slice(145, num_dims)

    return slices


pd.DataFrame([
    {"metric_group": name, "start": start, "stop": stop}
    for name, (start, stop) in BASE_SLICES.items()
])


## File Discovery

In [ ]:
def parse_decoder_eval_filename(path, task_prefix=TASK_PREFIX, filename_suffix=FILENAME_SUFFIX):
    stem = Path(path).stem

    if filename_suffix and stem.endswith(filename_suffix):
        stem = stem[:-len(filename_suffix)]

    prefix = f"{task_prefix}_"
    if not stem.startswith(prefix):
        return None

    remainder = stem[len(prefix):]
    parts = remainder.split("_")
    if len(parts) < 2:
        return None

    terrain = parts[0]
    disturbance_condition = "_".join(parts[1:])

    return {
        "task": task_prefix,
        "terrain": terrain,
        "disturbance_condition": disturbance_condition,
        "path": Path(path),
    }


def find_decoder_eval_files(exp_folder, task_prefix=TASK_PREFIX, disturbance_condition=DISTURBANCE_CONDITION):
    exp_folder = Path(exp_folder)
    records = []

    pattern = f"{task_prefix}_*{FILENAME_SUFFIX}.csv" if FILENAME_SUFFIX else f"{task_prefix}_*.csv"
    for path in sorted(exp_folder.rglob(pattern)):
        if EXCLUDE_DIRS:
            rel_parts = path.relative_to(exp_folder).parts
            if any(part in EXCLUDE_DIRS for part in rel_parts):
                continue

        info = parse_decoder_eval_filename(path, task_prefix=task_prefix)
        if info is None:
            continue

        if disturbance_condition is not None and info["disturbance_condition"] != disturbance_condition:
            continue

        records.append(info)

    return pd.DataFrame(records)


file_table = find_decoder_eval_files(EXP_FOLDER)
file_table


## Streaming Metric Utilities

In [ ]:
def parse_array(value):
    if isinstance(value, str):
        return ast.literal_eval(value)
    return value


def to_2d_float_array(value):
    arr = np.asarray(parse_array(value), dtype=np.float64)
    if arr.ndim == 1:
        arr = arr.reshape(1, -1)
    return arr


def to_1d_bool_success(value, num_envs):
    arr = np.asarray(parse_array(value))
    if arr.ndim == 0:
        arr = np.full(num_envs, arr.item())
    arr = arr.reshape(-1)
    if arr.size == 1 and num_envs != 1:
        arr = np.full(num_envs, arr.item())
    if arr.size != num_envs:
        raise ValueError(f"failure length {arr.size} does not match num_envs {num_envs}")
    return arr.astype(float) == 0.0


class RunningErrorStats:
    def __init__(self):
        self.n_rows_total = 0
        self.n_samples_total = 0
        self.n_samples_success = 0
        self.n_samples_failed = 0
        self.groups = {}
        self.num_dims_seen = set()

    def update(self, pred, gt, success_mask):
        if pred.shape != gt.shape:
            raise ValueError(f"pred shape {pred.shape} does not match gt shape {gt.shape}")

        self.n_rows_total += 1
        self.n_samples_total += int(pred.shape[0])
        self.n_samples_success += int(success_mask.sum())
        self.n_samples_failed += int((~success_mask).sum())
        self.num_dims_seen.add(int(pred.shape[1]))

        if not success_mask.any():
            return

        err = pred[success_mask] - gt[success_mask]
        abs_err = np.abs(err)
        sq_err = np.square(err)
        slices = effective_slices(err.shape[1])

        for name, slc in slices.items():
            part_abs = abs_err[:, slc]
            part_sq = sq_err[:, slc]
            rec = self.groups.setdefault(
                name,
                {"n_samples": 0, "n_values": 0, "sum_abs": 0.0, "sum_sq": 0.0, "sum_l2": 0.0},
            )
            rec["n_samples"] += int(part_abs.shape[0])
            rec["n_values"] += int(part_abs.size)
            rec["sum_abs"] += float(part_abs.sum())
            rec["sum_sq"] += float(part_sq.sum())
            rec["sum_l2"] += float(np.linalg.norm(err[success_mask][:, slc], axis=1).sum())

    def merge(self, other):
        self.n_rows_total += other.n_rows_total
        self.n_samples_total += other.n_samples_total
        self.n_samples_success += other.n_samples_success
        self.n_samples_failed += other.n_samples_failed
        self.num_dims_seen.update(other.num_dims_seen)

        for name, src in other.groups.items():
            dst = self.groups.setdefault(
                name,
                {"n_samples": 0, "n_values": 0, "sum_abs": 0.0, "sum_sq": 0.0, "sum_l2": 0.0},
            )
            for key in dst:
                dst[key] += src[key]

    def to_rows(self):
        rows = []
        failure_rate = self.n_samples_failed / self.n_samples_total if self.n_samples_total else np.nan
        dims_seen = ",".join(map(str, sorted(self.num_dims_seen)))

        for name, rec in sorted(self.groups.items()):
            n_values = rec["n_values"]
            n_samples = rec["n_samples"]
            rows.append({
                "metric_group": name,
                "mae": rec["sum_abs"] / n_values if n_values else np.nan,
                "rmse": math.sqrt(rec["sum_sq"] / n_values) if n_values else np.nan,
                "mean_l2_error": rec["sum_l2"] / n_samples if n_samples else np.nan,
                "num_success_samples": n_samples,
                "num_total_samples": self.n_samples_total,
                "num_failed_samples": self.n_samples_failed,
                "num_scalar_values": n_values,
                "sum_abs_error": rec["sum_abs"],
                "sum_sq_error": rec["sum_sq"],
                "sum_l2_error": rec["sum_l2"],
                "failure_rate": failure_rate,
                "num_csv_rows": self.n_rows_total,
                "priv_obs_dims_seen": dims_seen,
            })
        return rows


## Multiprocessing File Analysis

Each worker reads only the required columns and accumulates scalar sums per chunk. The full CSV is never concatenated into one large dataframe.


In [ ]:
def analyze_decoder_file(record):
    path = Path(record["path"])
    stats = RunningErrorStats()

    converters = {col: parse_array for col in USECOLS}
    reader = pd.read_csv(path, usecols=USECOLS, converters=converters, chunksize=CHUNKSIZE)

    for chunk in reader:
        for row in chunk.itertuples(index=False):
            row_dict = row._asdict()
            pred = to_2d_float_array(row_dict["decoder_pred_priv_obs"])
            gt = to_2d_float_array(row_dict["gt_priv_obs_latest"])
            success = to_1d_bool_success(row_dict["failure"], pred.shape[0])
            stats.update(pred, gt, success)

    rows = stats.to_rows()
    for out in rows:
        out.update({
            "task": record["task"],
            "terrain": record["terrain"],
            "disturbance_condition": record["disturbance_condition"],
            "path": str(path),
        })
    return rows


def analyze_files_parallel(file_table, max_workers=NUM_WORKERS):
    records = file_table.to_dict("records")
    if not records:
        return pd.DataFrame()

    all_rows = []
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(analyze_decoder_file, record): record for record in records}
        for future in as_completed(futures):
            record = futures[future]
            try:
                all_rows.extend(future.result())
            except Exception as exc:
                print(f"Failed: {record['path']} -> {exc}")

    return pd.DataFrame(all_rows)


file_results = analyze_files_parallel(file_table)
file_results.head()


## Aggregate Results

In [ ]:
def aggregate_result_rows(df, by):
    if df.empty:
        return df.copy()

    group_cols = list(by) + ["metric_group"]
    rows = []

    for keys, group in df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        key_data = dict(zip(group_cols, keys))

        n_success = int(group["num_success_samples"].sum())
        n_total = int(group["num_total_samples"].sum())
        n_failed = int(group["num_failed_samples"].sum())
        n_values = int(group["num_scalar_values"].sum())
        sum_abs = float(group["sum_abs_error"].sum())
        sum_sq = float(group["sum_sq_error"].sum())
        sum_l2 = float(group["sum_l2_error"].sum())

        rows.append({
            **key_data,
            "mae": sum_abs / n_values if n_values else np.nan,
            "rmse": math.sqrt(sum_sq / n_values) if n_values else np.nan,
            "mean_l2_error": sum_l2 / n_success if n_success else np.nan,
            "num_success_samples": n_success,
            "num_total_samples": n_total,
            "num_failed_samples": n_failed,
            "num_scalar_values": n_values,
            "sum_abs_error": sum_abs,
            "sum_sq_error": sum_sq,
            "sum_l2_error": sum_l2,
            "failure_rate": n_failed / n_total if n_total else np.nan,
            "num_files": int(group["path"].nunique()) if "path" in group else np.nan,
            "priv_obs_dims_seen": ",".join(sorted(set(",".join(group["priv_obs_dims_seen"].dropna()).split(",")) - {""})),
        })

    return pd.DataFrame(rows)


per_terrain = aggregate_result_rows(file_results, ["terrain"])
per_disturbance = aggregate_result_rows(file_results, ["disturbance_condition"])
per_terrain_disturbance = aggregate_result_rows(file_results, ["terrain", "disturbance_condition"])

all_results = aggregate_result_rows(
    file_results.assign(all_results="all"),
    ["all_results"],
)

per_terrain.to_csv(RESULTS_DIR / "decoder_error_per_terrain.csv", index=False)
per_disturbance.to_csv(RESULTS_DIR / "decoder_error_per_disturbance.csv", index=False)
per_terrain_disturbance.to_csv(RESULTS_DIR / "decoder_error_per_terrain_disturbance.csv", index=False)
all_results.to_csv(RESULTS_DIR / "decoder_error_all_results.csv", index=False)
file_results.to_csv(RESULTS_DIR / "decoder_error_per_file.csv", index=False)

print(f"Saved result tables to {RESULTS_DIR}")
print(f"Analyzed {file_results['path'].nunique() if not file_results.empty else 0} files")
print(f"Metric groups: {sorted(file_results['metric_group'].unique()) if not file_results.empty else []}")


## Tables

In [ ]:
display_cols = [
    "metric_group",
    "mae",
    "rmse",
    "mean_l2_error",
    "num_success_samples",
    "failure_rate",
    "priv_obs_dims_seen",
]

print("All results")
display(all_results[display_cols] if not all_results.empty else all_results)

print("Per terrain")
display(per_terrain[["terrain", *display_cols]] if not per_terrain.empty else per_terrain)

print("Per disturbance condition")
display(per_disturbance[["disturbance_condition", *display_cols]] if not per_disturbance.empty else per_disturbance)


## Plots

In [ ]:
def plot_metric(results, x, metric="rmse", metric_groups=None, title=None):
    if results.empty:
        print("No results to plot.")
        return

    plot_df = results.copy()
    if metric_groups is not None:
        plot_df = plot_df[plot_df["metric_group"].isin(metric_groups)]

    if plot_df.empty:
        print("No rows after metric_group filtering.")
        return

    plt.figure(figsize=(max(8, 0.45 * plot_df[x].nunique() * plot_df["metric_group"].nunique()), 5))
    if sns is not None:
        sns.barplot(data=plot_df, x=x, y=metric, hue="metric_group")
    else:
        for metric_group, group in plot_df.groupby("metric_group"):
            plt.plot(group[x], group[metric], marker="o", label=metric_group)
        plt.legend()
    plt.xticks(rotation=35, ha="right")
    plt.ylabel(metric)
    plt.title(title or f"Decoder prediction {metric} by {x}")
    plt.tight_layout()
    plt.show()


KEY_GROUPS = [
    "entire_privileged_observation",
    "standard_observation_vector",
    "base_velocity",
    "domain_randomization_all",
    "base_height",
    "terrain_shape_around_feet",
    "feet_contact_state_and_height",
    "ground_reaction_forces",
]

plot_metric(per_terrain, "terrain", metric="rmse", metric_groups=KEY_GROUPS, title="Decoder RMSE per terrain")
plot_metric(per_disturbance, "disturbance_condition", metric="rmse", metric_groups=KEY_GROUPS, title="Decoder RMSE per disturbance")


In [ ]:
DOMAIN_GROUPS = [
    "domain_friction",
    "domain_added_mass",
    "domain_com_shift",
    "domain_pushes",
    "domain_wrenches",
]

plot_metric(per_terrain, "terrain", metric="rmse", metric_groups=DOMAIN_GROUPS, title="Domain-randomization RMSE per terrain")
plot_metric(per_disturbance, "disturbance_condition", metric="rmse", metric_groups=DOMAIN_GROUPS, title="Domain-randomization RMSE per disturbance")


## Inspect One Metric Group

In [ ]:
METRIC_GROUP_TO_INSPECT = "entire_privileged_observation"

inspect = per_terrain_disturbance[
    per_terrain_disturbance["metric_group"] == METRIC_GROUP_TO_INSPECT
].sort_values(["terrain", "disturbance_condition"])

inspect
